# Análise Exploratória de Dados (EDA)
## Oxford Parkinson's Disease Telemonitoring Dataset

**Disciplina:** Ciência de Dados — Centro Universitário Dom Helder  
**Orientador:** Prof. Dr. Marcos W. Rodrigues  
**Objetivo:** Investigar a distribuição e as relações entre os atributos biomédicos de voz e os estágios de progressão da Doença de Parkinson.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

# Limiares clínicos de discretização (UPDRS Total)
BINS   = [0, 25, 40, 100]
LABELS_MAP = {0: 'Leve', 1: 'Moderado', 2: 'Grave'}
COLORS = ['#4C72B0', '#DD8452', '#55A868']
print('Ambiente configurado.')

## 1. Carregamento e Visão Geral

In [ ]:
df = pd.read_csv('../data/raw/parkinsons_telemonitoring.csv')
df['progressao'] = pd.cut(df['total_UPDRS'], bins=BINS, labels=[0,1,2]).astype(int)
df['progressao_label'] = df['progressao'].map(LABELS_MAP)

print(f'Dimensões do dataset: {df.shape[0]} registros × {df.shape[1]} atributos')
print(f'\nValores nulos por coluna:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
print(f'\nSem valores nulos!' if df.isnull().sum().sum()==0 else '')
df.head(3)

## 2. Definição e Justificativa da Variável-Alvo

A coluna `total_UPDRS` (Unified Parkinson's Disease Rating Scale — parte III) é uma medida contínua da gravidade dos sintomas motores. Para o problema de **classificação multiclasse**, ela foi discretizada em três faixas com base em limiares clínicos da literatura (Goetz *et al.*, 2008; MDS):

| Classe | Faixa UPDRS | Interpretação Clínica |
|--------|-------------|----------------------|
| 0 — Leve | ≤ 25 | Sintomas leves/iniciais, menor comprometimento motor |
| 1 — Moderado | 26–40 | Comprometimento funcional moderado |
| 2 — Grave | > 40 | Comprometimento grave, maior dependência |


In [ ]:
order = ['Leve', 'Moderado', 'Grave']
counts = df['progressao_label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
bars = axes[0].bar(order, [counts[o] for o in order], color=COLORS, edgecolor='white', width=0.6)
axes[0].set_title('Distribuição das Classes de Progressão', fontweight='bold')
axes[0].set_ylabel('Número de Registros')
for bar, val in zip(bars, [counts[o] for o in order]):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 30,
                  f'{val}\n({val/len(df):.1%})', ha='center', va='bottom', fontsize=10)

axes[1].pie([counts[o] for o in order], labels=order, colors=COLORS,
             autopct='%1.1f%%', startangle=90,
             wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Proporção das Classes', fontweight='bold')

plt.suptitle('Figura 1 — Variável-Alvo: Estágio de Progressão do Parkinson\n'
             '(total_UPDRS discretizado em 3 classes)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('data/fig01_distribuicao_classes.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Distribuição do Total UPDRS com os Limiares

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['total_UPDRS'], bins=60, color='#4C72B0', edgecolor='white', alpha=0.8)
ax.axvline(25, color='red',    linestyle='--', linewidth=2, label='Limiar Leve→Moderado (25)')
ax.axvline(40, color='orange', linestyle='--', linewidth=2, label='Limiar Moderado→Grave (40)')
ax.set_xlabel('Total UPDRS'); ax.set_ylabel('Frequência')
ax.set_title('Figura 2 — Distribuição do Total UPDRS com Limiares de Discretização',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
for x, label, color in [(12.5,'Leve\n(≤25)','#4C72B0'),
                         (32.5,'Moderado\n(26–40)','#DD8452'),
                         (47,'Grave\n(>40)','#55A868')]:
    ax.text(x, ax.get_ylim()[1]*0.85, label, ha='center', fontsize=11,
             color=color, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig05_updrs_discretizacao.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Estatísticas Descritivas por Classe

In [ ]:
feature_cols = [c for c in df.columns if c not in ['motor_UPDRS','total_UPDRS','progressao','progressao_label']]

desc = df.groupby('progressao_label')[feature_cols].mean().T
desc.columns.name = 'Classe'
desc[['Leve','Moderado','Grave']].style\
    .format('{:.4f}')\
    .background_gradient(cmap='Blues', axis=1)\
    .set_caption('Tabela 1 — Médias dos Atributos de Voz por Classe de Progressão')

## 5. Matriz de Correlação

In [ ]:
voice_features = ['Jitter(%)','Jitter(Abs)','Jitter:RAP','Jitter:PPQ5','Shimmer',
                   'Shimmer(dB)','Shimmer:APQ3','NHR','HNR','RPDE','DFA','PPE']

corr = df[voice_features + ['progressao']].corr()
fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.4, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Figura 3 — Matriz de Correlação: Atributos de Voz e Variável-Alvo',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('data/fig02_heatmap_correlacao.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Boxplots dos Atributos Mais Relevantes por Classe

In [ ]:
key_features = ['Jitter(%)','Shimmer','HNR','RPDE','DFA','PPE']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()
for i, feat in enumerate(key_features):
    sns.boxplot(data=df, x='progressao_label', y=feat, order=order,
                palette={'Leve':'#4C72B0','Moderado':'#DD8452','Grave':'#55A868'},
                ax=axes[i])
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_xlabel('Estágio de Progressão')
plt.suptitle('Figura 4 — Boxplots dos Atributos de Voz por Estágio de Progressão',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('data/fig03_boxplots_por_classe.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Testes de Significância Estatística (Kruskal-Wallis)

In [ ]:
results = []
for feat in feature_cols:
    groups = [df[df['progressao']==c][feat].dropna().values for c in [0,1,2]]
    stat, pval = stats.kruskal(*groups)
    results.append({'Atributo': feat,
                    'Média Leve'    : df[df['progressao']==0][feat].mean(),
                    'Média Moderado': df[df['progressao']==1][feat].mean(),
                    'Média Grave'   : df[df['progressao']==2][feat].mean(),
                    'H (Kruskal)'   : round(stat, 2),
                    'p-valor'       : pval,
                    'Significativo' : '✓' if pval < 0.05 else '✗'})

sig_df = pd.DataFrame(results).sort_values('H (Kruskal)', ascending=False).reset_index(drop=True)
print('Tabela 2 — Teste de Kruskal-Wallis por Atributo')
sig_df.style.format({'Média Leve':'{:.4f}','Média Moderado':'{:.4f}',
                      'Média Grave':'{:.4f}','p-valor':'{:.2e}'})\
            .background_gradient(subset=['H (Kruskal)'], cmap='Reds')

## 8. Pairplot dos Atributos Mais Discriminativos

In [ ]:
top_feats = ['HNR','PPE','DFA','RPDE','Jitter(%)']
pair_df = df[top_feats + ['progressao_label']].copy()

g = sns.pairplot(pair_df, hue='progressao_label', hue_order=order,
                  plot_kws={'alpha': 0.35, 's': 12},
                  palette={'Leve':'#4C72B0','Moderado':'#DD8452','Grave':'#55A868'},
                  diag_kind='kde')
g.fig.suptitle('Figura 5 — Pair Plot: Top 5 Atributos × Estágio de Progressão',
               y=1.02, fontsize=13, fontweight='bold')
plt.savefig('data/fig06_pairplot.png', bbox_inches='tight', dpi=120)
plt.show()

## 9. Sumário da EDA

| Aspecto | Observação |
|---------|------------|
| **Dimensões** | 5.875 registros × 19 features de voz + variável-alvo |
| **Valores nulos** | Nenhum em toda a base |
| **Balanceamento** | Leve 37,2% · Moderado 45,6% · Grave 17,1% — desbalanceado → SMOTE |
| **Outliers** | Presentes em Jitter e Shimmer — tratados via IQR × 3 |
| **Features mais correlacionadas** | HNR (↓), PPE (↑), DFA (↓), RPDE (↑) com a progressão |
| **Multicolinearidade** | Alta entre métricas de Jitter e entre métricas de Shimmer |
| **Significância estatística** | Todos os atributos são estatisticamente significativos (p < 0,05) pelo teste de Kruskal-Wallis |